### Top-Spin

Implementiere folgende FUnktionen.
Schreibe zu jeder Funktionen einen Test.

- Erstelle eigene Implementationen der Suchstrategien.
  Arbeite mit einer get_neighbor Funktion, die den neuen Knoten und den Namen der Operation zurück gibt.
- Schreibe `get_path_to_goal(node, go_back)`
- Schreibe `follow_path(start, path)`
- Schreibe `reverse_path(start, path)`

In [257]:
import pickle

# Save the dictionary
with open("go_back6.pkl", "wb") as f:
    pickle.dump(go_back6, f, protocol=pickle.HIGHEST_PROTOCOL)

# Load the dictionary
# with open("go_back6.pkl", "rb") as f:
#     loaded_dict = pickle.load(f)


In [266]:
import importlib
import searchstrategies as S
import random
importlib.reload(S)


def reset(numbers):
    return tuple(sorted(numbers))


def shift_left(numbers):
    return numbers[1:] + numbers[:1]


def shift_right(numbers):
    return numbers[-1:] + numbers[:-1]


def shift_nleft(numbers, i):
    return numbers[i:] + numbers[:i]


def shift_nright(numbers, i):
    return numbers[-i:] + numbers[:-i]


def swap4(numbers):
    return numbers[3::-1] + numbers[4:]

def gswap4(numbers, i):
    n = len(numbers)
    i1, i2, i3 = (i+1) % n, (i+2) % n, (i+3) % n
    ns = list(numbers)
    ns[i], ns[i3] = ns[i3], ns[i]
    ns[i1], ns[i2] = ns[i2], ns[i1]
    return tuple(ns)


def gswap(numbers, i):
    if i == 0:
        return shift_nleft(gswap4(numbers, i), 3)
    elif i == 19:
        return shift_left(gswap4(numbers, i))
    elif i == 18:
        return shift_right(gswap4(numbers, i))
    elif i == 17:
        return shift_nright(gswap4(numbers, i), 3)

    return gswap4(numbers, i)


def scramble(numbers):
    numbers = list(numbers)
    random.shuffle(numbers)
    return tuple(numbers)


def get_neighbors2(numbers):
    ops = (shift_left, shift_right, swap4)
    labels = (-1, 1, 0)
    for op, label in zip(ops, labels):
        yield label, op(numbers)


def get_neighbors(numbers):
    for i in range(len(numbers)-1):
        yield i, shift_nright(numbers, i)
    yield 0, swap4(numbers)



def follow_path(numbers, path):
    for i in path:
        if i == 0:
            numbers = swap4(numbers)
        elif i == -1:
            numbers = shift_left(numbers)
        elif i == 1:
            numbers = shift_right(numbers)
        elif i < -1:
            numbers = shift_nleft(numbers, i)
        elif i > 1:
            numbers = shift_nright(numbers, i)

    return numbers


def follow_path1(numbers, path, h):
    print(numbers, h(numbers))
    for i in path:
        if i == 0:
            numbers = swap4(numbers)
        elif i == -1 :
            numbers = shift_left(numbers,)
        elif i == 1:
            numbers = shift_right(numbers)
        print(numbers, h(numbers))
    return numbers


def is_path(numbers, targets, path):
    return targets == follow_path(numbers, path)


def reverse_path(path):
    tt = {'l': 'r', 'r': 'l'}
    return ''.join(tt.get(op, op) for op in reversed(path))


def n_matches(numbers, goal):
    return sum(x == y for x, y in zip(numbers, goal))


def align(numbers):
    i = numbers.index(0)
    xs = numbers[i:] + numbers[:i]
    return xs


def get_inversions(numbers, targets):
    inversions = 0
    xs, _ = align(numbers, targets)
    n = len(numbers)

    for i in range(n):
        for j in range(i + 1, n):
            if (xs[i] < xs[j]) != (targets[i] < targets[j]):
                inversions += 1

    return inversions

In [267]:
xs = tuple(range(20))
gswap(xs, 18)

(0, 19, 18, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 1)

In [268]:
def get_pairs(numbers):
    pairs = set((numbers[i-1], numbers[i]) for i in range(len(numbers)))
    return pairs


def get_bad_pairs(numbers, pairs):
    return sum((numbers[i-1], numbers[i]) not in pairs for i in range(len(numbers)))


def circular_dist(a, b, n):
    '''Sei 0<= a,b < n. 0..n-1 sind auf einen Kreis angeordnet
       Berechnet die von a nach b
    '''
    d = abs(b-a)
    return min(d, n-d)


def longest_run_plus_shift(numbers):
    bps = []
    n = len(numbers)

    for i in range(n):
        if (numbers[i-1] + 1) % n != numbers[i]:
            bps.append(i)

    m = len(bps)
    if m == 0:
        longest = n
        start = 0
    else:
        longest, start = max(((bps[i] - bps[i-1]) % n, bps[i-1]) for i in range(m))

    shift = circular_dist(numbers[start], start, n)
    return longest, shift


def longest_run(numbers):
    bps = []
    n = len(numbers)

    for i in range(n):
        if (numbers[i-1] + 1) % n != numbers[i]:
            bps.append(i)

    m = len(bps)
    if m == 0:
        longest = n
    else:
        longest = max((bps[i] - bps[i-1]) % n for i in range(m))

    return longest



def initial_len(numbers):
    n = len(numbers)
    i = numbers.index(0)
    for j in range(n):
        if numbers[(i+j) % n] != j:
            return n - j
    return 0


def build_instruction(numbers):
    tt = (0, 3, 2)
    n = len(numbers)
    i = numbers.index(0)

    for j in range(n):
        if numbers[k := (i+j) % n] == j:
            continue

        length = j
        idx = numbers.index(j)
        d = (idx - k + 1) % n - 1
        penalty = d//3 + tt[d % 3]
        d_wheel = (3 - idx) % n
        cost = 100*(n - length) + 10*penalty + d_wheel
        #print(length, d, penalty, d_wheel)
        return cost

    return 0


def dist(numbers):
    return sum(abs(numbers[i]-i) for i in range(len(numbers)))


def max_dist(numbers):
    return max(abs(numbers[i]-i) for i in range(len(numbers)))


def sign(numbers):
    '''sign of the permutation, i.e. number of swaps % 2'''
    sign = 0
    n = len(numbers)
    for i in range(n-1):
        for j in range(i+1, n):
            sign += (numbers[i] > numbers[j])
    return sign % 2

def canonic(state):
    i = state.index(0)
    return state[i:] + state[:i]

In [269]:
canonic((5,6,0,7,1,2,3,4))

(0, 7, 1, 2, 3, 4, 5, 6)

In [270]:
longest_run_plus_shift((6,0,7,1,2,3,4,5))

(6, 2)

In [5]:
longest_run_plus_shift((5,6,0,7,1,2,3,4))

(6, 3)

In [6]:
longest_run((2,3,4,5,6,7,0,1))

8

In [7]:
build_instruction((0,1,2,3,7,6,5,4))

414

In [8]:
build_instruction((4,0,2,1,3,7,6,5))

730

In [9]:
build_instruction((6,0,1,2,3,4,7,5))

334

In [10]:
scramble_1 = (4, 8, 0, 1, 9, 19, 15, 2, 17, 16, 5, 7, 18, 14, 3, 12, 10, 6, 11, 13)

In [11]:
build_instruction(scramble_1)

1826

In [12]:
longest_run_plus_shift(scramble_1)

(2, 2)

In [271]:
start = tuple(range(10))


node, go_back, dd = S.search_bf(start, get_neighbors2, None)
path = S.get_path_to_goal(node, go_back)
len(path)

KeyboardInterrupt: 

In [111]:
start = tuple(range(20))


node, go_back, dd = S.search_bf(start, get_neighbors2, None, max_depth=20)
path = S.get_path_to_goal(node, go_back)
len(path)

Failure. Count: 2612876


20

In [243]:
start = tuple(range(20))
pairs = get_pairs(goal)


def h(numbers):
    return get_bad_pairs(numbers, pairs)


node, go_back6 = S.search_bf_2(start, get_neighbors2, h, 7)

Failure. Count: 60859639


In [231]:
goal = tuple(range(20))
start = tuple(range(18)) + (19, 18)
pairs = get_pairs(goal)

def h(numbers):
    return get_bad_pairs(numbers, pairs)


node, go_back = S.search_greedy(start, get_neighbors2, h, goal)

path = S.get_path_to_goal(node, go_back)
print(len(path), is_path(start, goal, path))

Success. Count: 40582
41 True


In [183]:
start = tuple(range(20))
pairs = get_pairs(goal)


def h(numbers):
    return get_bad_pairs(numbers, pairs)

    
node, dd = S.search_bf_1(start, get_neighbors2, h, canonic, 11)
len(dd)

ignore (1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 0) (0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19) 0
ignore (19, 0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18) (0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19) 0
(3, 2, 1, 0, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19) 5
ignore (2, 1, 0, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 3) (0, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 3, 2, 1) 5
ignore (19, 3, 2, 1, 0, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18) (0, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 3, 2, 1) 5
ignore (0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19) (0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19) 0
Failure. Count: 2


2

In [170]:
dd

{(0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19): 0,
 (0, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 3, 2, 1): 1}

In [184]:
goal = tuple(range(20))


pairs = get_pairs(goal)


def h(numbers):
    return get_bad_pairs(numbers, pairs)

# does not yield shortes paths!! 
node, go_back6 = S.search_greedy_1(goal, get_neighbors, h, None, threshold=7)

Failure. Count: 30429819


In [225]:
goal = tuple(range(20))
start = scramble(goal)
# start = (10, 0, 9, 4, 16, 8, 14, 1, 2, 17, 19, 11, 5, 6, 3, 13, 18, 12, 7, 15)

pairs = get_pairs(goal)


def h(numbers):
    return get_bad_pairs(numbers, pairs)


node, go_back = S.search_greedy(start, get_neighbors2, h, None, threshold=5)

Success. Count: 0


In [249]:
node in go_back6, h(node)

(True, 4)

In [236]:
h(node), node

(3, (0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 19, 18))

In [264]:
path = S.get_path_to_goal(node, go_back6)
print(len(path))

47


In [244]:
node = tuple(range(18)) + (19, 18)

In [248]:
node = (2, 1, 0) + tuple(range(3,20))

In [251]:
node = (3, 1, 2, 0) + tuple(range(4,20))

In [253]:
node = (1, 0, 2, 3) + tuple(range(4,20))

In [263]:
node = tuple(range(14)) + (19, 18, 17, 16, 15, 14)
h(node)

7

In [128]:
goal = tuple(range(20))
start = node


pairs = get_pairs(goal)


def h(numbers):
    return get_bad_pairs(numbers, pairs)



node, go_back = S.search_greedy(start, get_neighbors2, h, goal)
path = S.get_path_to_goal(node, go_back)
print(len(path), is_path(start, goal, path))

Success. Count: 94514
141 True


In [129]:
start

(8, 9, 2, 1, 16, 17, 18, 19, 0, 10, 11, 12, 13, 14, 15, 3, 4, 5, 6, 7)

In [ ]:
follow_path1(start, path, h)

1970